# 02 · Feature engineering & enrichment
Every feature must be **known at dispatch time**. Anything known only after delivery (actual arrival, GPS pings, delay flags) is banned – see `config.LEAKY_COLUMNS`.

In [1]:
# Setup: make the project importable from the notebooks/ folder
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault("ARRIVO_OFFLINE", "1")     # use cached/offline enrichment for reproducibility
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)

def show(path, width=11):
    """Display a saved figure inline."""
    img = plt.imread(path); h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w)); ax.imshow(img); ax.axis("off"); plt.show()

In [2]:
from src.data.load import load_canonical
from src.data.clean import clean
from src.data.enrich import enrich, calendar_features, weather_for_location, route_info
canonical, source = load_canonical(); df, _ = clean(canonical)
print(source, len(df))

synthetic 14892


## 1. Weather (Open-Meteo, cached, with climatology fallback)
One API request per location covers the whole date range; results are cached in `data/interim/weather_cache/`. Offline (or for the synthetic dataset) a deterministic climatology model based on regional normals is used.

In [3]:
from datetime import date
mumbai_july = weather_for_location(19.08, 72.88, 'West', 0, [date(2024,7,d) for d in range(1,8)], live=False)
mumbai_jan  = weather_for_location(19.08, 72.88, 'West', 0, [date(2024,1,d) for d in range(1,8)], live=False)
pd.concat({'July': mumbai_july.set_index('date'), 'January': mumbai_jan.set_index('date')}, axis=1)

July                                 January                                
           rain_mm temp_c wind_kmph weather_source rain_mm temp_c wind_kmph weather_source
date                                                                                      
2024-07-01    56.3   29.9      11.9    climatology     NaN    NaN       NaN            NaN
2024-07-02    37.5   26.7      10.1    climatology     NaN    NaN       NaN            NaN
2024-07-03    11.3   26.5      18.0    climatology     NaN    NaN       NaN            NaN
2024-07-04     1.0   27.2      10.3    climatology     NaN    NaN       NaN            NaN
2024-07-05    29.7   27.6      18.0    climatology     NaN    NaN       NaN            NaN
2024-07-06     0.0   27.9       3.7    climatology     NaN    NaN       NaN            NaN
2024-07-07    43.0   28.7      10.5    climatology     NaN    NaN       NaN            NaN
2024-01-01     NaN    NaN       NaN            NaN     0.0   22.7      14.6    climatology
2024-01-02     NaN    NaN       NaN            NaN     0.0   25.8      17.4    climatology
2024-01-03     NaN    NaN       NaN            NaN     0.0   24.3      13.9    climatology
2024-01-04     NaN    NaN       NaN            NaN     0.0   21.9       0.8    climatology
2024-01-05     NaN    NaN       NaN            NaN     0.0   27.3      15.0    climatology
2024-01-06     NaN    NaN       NaN            NaN     0.0   24.7      14.8    climatology
2024-01-07     NaN    NaN       NaN            NaN     0.0   26.1       8.7    climatology

## 2. Route (OSRM) and calendar (holidays + Indian festivals)

In [4]:
print(route_info(28.61, 77.21, 26.91, 75.79, live=False))   # Delhi -> Jaipur (offline estimate)
pd.DataFrame([calendar_features(pd.Timestamp(d)) | {'date': d} for d in ['2025-10-18','2025-10-20','2025-07-15','2025-01-26','2025-02-10']]).set_index('date')

{'osrm_km': 293.8238832411613, 'osrm_duration_hrs': 5.342252422566569, 'route_source': 'estimate'}


,is_national_holiday,is_festival_window,days_to_festival,nearest_festival,is_monsoon
date,,,,,
2025-10-18,0,1,2,Diwali,0
2025-10-20,0,1,0,Diwali,0
2025-07-15,0,0,30,Ganesh Chaturthi,1
2025-01-26,1,0,12,Pongal/Sankranti,0
2025-02-10,0,0,27,Pongal/Sankranti,0


## 3. Build the model frames
`build_model_frames` = row features → **time-based split** → route/supplier history fitted on **training data only** (out-of-fold for training rows).

In [5]:
from src.features.build_features import build_model_frames, feature_matrix
from src import config
enriched = enrich(df, live=(source != 'synthetic'))
train, val, test, hist = build_model_frames(enriched)
pd.DataFrame({s: [len(d), d.dispatch_time.min().date(), d.dispatch_time.max().date(), round(d.is_delayed.mean(),3)] for s, d in [('train',train),('validation',val),('test',test)]}, index=['rows','from','to','delay rate'])

,train,validation,test
rows,9126,2824,2942
from,2023-01-01,2025-01-01,2025-07-01
to,2024-12-31,2025-06-30,2025-12-30
delay rate,0.385,0.251,0.375


In [6]:
X = feature_matrix(train)
print(len(config.NUMERIC_FEATURES), 'numeric +', len(config.CATEGORICAL_FEATURES), 'categorical features')
X.head()

28 numeric + 4 categorical features


,road_distance_km,haversine_km,route_detour_ratio,osrm_duration_hrs,planned_hours,planned_speed_kmph,dispatch_hour,dispatch_weekday,dispatch_month,is_weekend,is_peak_hour,is_night_dispatch,is_national_holiday,is_festival_window,days_to_festival,is_monsoon,origin_is_hilly,dest_is_hilly,origin_rain_mm,dest_rain_mm,origin_temp_c,dest_temp_c,max_wind_kmph,is_heavy_rain,route_hist_delay_rate,route_hist_overrun_ratio,route_hist_trips,supplier_hist_delay_rate,vehicle_type,material_type,origin_region,dest_region
0,983.5,832.962293,1.180726,17.881818,36.393413,27.024121,5.0,6.0,1.0,1.0,0.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,21.3,15.1,12.0,0.0,0.358092,0.987905,9.0,0.255994,32 FT Multi-Axle,FMCG,Central,North
1,570.8,437.669775,1.304180,10.378182,19.768490,28.874233,5.0,6.0,1.0,1.0,0.0,0.0,0.0,0.0,14.0,0.0,1.0,0.0,0.0,0.0,0.6,16.7,15.0,0.0,0.541107,1.008481,11.0,0.691281,24 FT Open Body,Cement & Steel,North,North
2,636.0,502.593385,1.265436,11.563636,20.726831,30.684865,6.0,6.0,1.0,1.0,0.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,22.5,20.3,18.1,0.0,0.384609,0.981916,16.0,0.481139,20 FT Container,Electronics,South,South
3,1260.8,1097.505196,1.148787,22.923636,43.155641,29.215184,7.0,6.0,1.0,1.0,0.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,21.2,26.5,13.8,0.0,0.361187,0.960282,5.0,0.409666,32 FT Single-Axle,FMCG,West,South
4,987.9,890.967639,1.108794,17.961818,36.540378,27.035845,7.0,6.0,1.0,1.0,0.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,21.2,16.7,15.0,0.0,0.334128,0.953804,15.0,0.481139,32 FT Multi-Axle,Electronics,West,North


## 4. Why out-of-fold route history?
If a training row's own label were inside its route average, the model would learn to trust that feature too much (target leakage). Compare the in-sample vs out-of-fold correlation with the target:

In [7]:
from src.features.build_features import RouteHistory, base_features, time_split
b = base_features(enriched); tr, _, _ = time_split(b)
in_sample = RouteHistory().fit(tr).transform(tr)
print('corr(in-sample route rate, label):   ', round(in_sample.route_hist_delay_rate.corr(tr.is_delayed), 3))
print('corr(out-of-fold route rate, label):  ', round(train.route_hist_delay_rate.corr(train.is_delayed), 3))
print('corr(train-fitted rate, label) on VAL:', round(val.route_hist_delay_rate.corr(val.is_delayed), 3))

corr(in-sample route rate, label):    0.465
corr(out-of-fold route rate, label):   0.36
corr(train-fitted rate, label) on VAL: 0.328


The in-sample number is inflated; the out-of-fold number is close to what the model will really see on new data (validation).

## 5. Feature ↔ target relationships (Spearman)

In [8]:
num = train[config.NUMERIC_FEATURES + ['actual_hours','is_delayed']]
corr = num.corr(method='spearman')[['actual_hours','is_delayed']].drop(['actual_hours','is_delayed'])
corr.sort_values('is_delayed', ascending=False).round(3)

,actual_hours,is_delayed
route_hist_overrun_ratio,0.531,0.359
route_hist_delay_rate,0.568,0.358
planned_hours,0.960,0.253
road_distance_km,0.957,0.252
osrm_duration_hrs,0.957,0.252
haversine_km,0.952,0.244
dest_rain_mm,0.098,0.214
supplier_hist_delay_rate,0.050,0.211
is_monsoon,0.074,0.200
is_heavy_rain,0.053,0.188


Full descriptions of every feature: `docs/feature_list.md`.